# Deepfake Detector — Sequential Training (Kaggle, runs after you close your PC)

## How to use this notebook

1. **Attach all datasets** in the right sidebar (Input → Add Input → search for each dataset)
2. Set accelerator to **GPU T4 x2** or **P100** (Settings → Accelerator)
3. Set Internet to **On** (Settings → Internet → On) — needed for pip install
4. Click **Save Version** → **Save & Run All (Commit)**
5. **Close your browser / turn off your PC** — the notebook keeps running on Kaggle's servers for up to 9 hours
6. Come back later, check the output — checkpoints are saved to `/kaggle/working/`
7. If it didn't finish all datasets, just **Save Version & Run All** again — `--resume` picks up from the last checkpoint

**The sequential trainer:**
- Trains on ONE dataset at a time (no OOM, no 6h index wait)
- Indexes the next dataset in a background thread while training
- Saves a checkpoint after EVERY dataset (crash-proof)
- Carries model weights forward (continual learning)
- Stops after processing 35+ GB of data
- `--resume` skips already-completed datasets

In [ ]:
# Cell 1: Clone the repo and install dependencies
import os, sys, subprocess

if not os.path.isdir('/kaggle/working/AI_Deepfake_Detector'):
    os.chdir('/kaggle/working')
    # Replace YOUR_GITHUB_URL with your actual repo URL
    # Option A: clone from GitHub
    subprocess.run(['git', 'clone', 'https://github.com/your-username/AI_Deepfake_Detector.git'], check=False)
    
    # Option B: if you uploaded the code as a Kaggle dataset, uncomment:
    # subprocess.run(['cp', '-r', '/kaggle/input/your-code-dataset/AI_Deepfake_Detector', '/kaggle/working/'])

os.chdir('/kaggle/working/AI_Deepfake_Detector')
sys.path.insert(0, '/kaggle/working/AI_Deepfake_Detector')

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'librosa', 'tqdm'], check=False)

import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'GPU count: {torch.cuda.device_count()}')
print(f'Setup complete!')

In [ ]:
# Cell 2: Check what datasets are attached
import os

input_dir = '/kaggle/input'
if os.path.isdir(input_dir):
    print('Attached datasets:')
    for d in sorted(os.listdir(input_dir)):
        full = os.path.join(input_dir, d)
        if os.path.isdir(full):
            # Count files (quick estimate)
            try:
                count = sum(len(files) for _, _, files in os.walk(full))
                print(f'  {d}: ~{count} files')
            except Exception:
                print(f'  {d}: (error counting)')
else:
    print('No datasets attached! Add them in the right sidebar.')

In [ ]:
# Cell 3: VISION TRAINING — Sequential dataset-by-dataset
# This runs for several hours. Close your browser — it keeps running.
import subprocess, sys, os

output_dir = '/kaggle/working/checkpoints'
os.makedirs(output_dir, exist_ok=True)

cmd = [
    sys.executable, '-m', 'backend.training.train_sequential_kaggle',
    '--modality', 'vision',
    '--output-dir', output_dir,
    '--arch', 'cnn',
    '--backbone', 'efficientnet_b4',
    '--pretrained',
    '--epochs-per-dataset', '1',
    '--batch-size', '32',
    '--amp',
    '--resume',
    '--save-every', '500',
    '--target-gb', '35',
    '--workers', '4',
]

print('Starting VISION sequential training...')
print(f'Command: {" ".join(cmd)}')
print(f'Output dir: {output_dir}')
print(f'You can close your browser now — this keeps running.\n')

result = subprocess.run(cmd, cwd='/kaggle/working/AI_Deepfake_Detector')
print(f'\nVision training exit code: {result.returncode}')

In [ ]:
# Cell 4: AUDIO TRAINING — Sequential dataset-by-dataset
import subprocess, sys, os

cmd = [
    sys.executable, '-m', 'backend.training.train_sequential_kaggle',
    '--modality', 'audio',
    '--output-dir', output_dir,
    '--epochs-per-dataset', '2',
    '--batch-size', '64',
    '--amp',
    '--resume',
    '--save-every', '100',
    '--target-gb', '35',
    '--workers', '4',
]

print('Starting AUDIO sequential training...\n')
result = subprocess.run(cmd, cwd='/kaggle/working/AI_Deepfake_Detector')
print(f'\nAudio training exit code: {result.returncode}')

In [ ]:
# Cell 5: Verify checkpoints
import os

ckpt_dir = '/kaggle/working/checkpoints'
print('Checkpoints saved:')
for f in sorted(os.listdir(ckpt_dir)):
    if f.endswith('.pth'):
        size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / (1024*1024)
        print(f'  {f}: {size_mb:.1f} MB')

# These files are automatically saved as notebook output when you commit.
# You can download them from the notebook's Output tab.

## Resuming after a session timeout

If the notebook timed out at 9 hours and didn't finish all datasets:

1. Go to the notebook's **Output** tab → download `seq_checkpoint.pth` and `vision_best.pth`
2. Create a new Kaggle dataset from those checkpoint files (Add Data → New Dataset)
3. In a new notebook, attach that checkpoint dataset + the training datasets
4. Copy the checkpoints to `/kaggle/working/checkpoints/`
5. Run the same cells again — `--resume` picks up from the last completed dataset

```python
# In the new notebook, before running training:
import os, shutil
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
# Copy saved checkpoints from the attached dataset
for f in os.listdir('/kaggle/input/your-checkpoint-dataset'):
    if f.endswith('.pth'):
        shutil.copy(f'/kaggle/input/your-checkpoint-dataset/{f}', '/kaggle/working/checkpoints/')
```
Then run the training cells — it resumes automatically.